In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName('capstone 02').getOrCreate()

from google.colab import files

uploaded = files.upload()

Saving payments.csv to payments.csv
Saving plans.json to plans.json
Saving usage.csv to usage.csv
Saving customers.csv to customers.csv


In [7]:
from pyspark.sql.functions import *

customers_df = spark.read.csv(
    'customers.csv',
    header = True,
    inferSchema = True
)

usage_df = spark.read.csv(
    'usage.csv',
    header = True,
    inferSchema = True
)

payments_df = spark.read.csv(
    'payments.csv',
    header = True,
    inferSchema = True
)

plans_df = spark.read.option(
    'multiline',
    'true'
).json('plans.json')

In [3]:
customers_df.printSchema()
usage_df.printSchema()
payments_df.printSchema()
plans_df.printSchema()

print('customer rec count : ', customers_df.count())
print('usage rec count : ', usage_df.count())
print('payments rec count : ', payments_df.count())
print('plans rec count : ', plans_df.count())

!mkdir -p bronze
customers_df.write.mode('overwrite').parquet('bronze/customers')
usage_df.write.mode('overwrite').parquet('bronze/usage')
payments_df.write.mode('overwrite').parquet('bronze/payments')
plans_df.write.mode('overwrite').parquet('bronze/plans')

root
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- plan_id: string (nullable = true)
 |-- status: string (nullable = true)

root
 |-- usage_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- usage_month: timestamp (nullable = true)
 |-- data_used_gb: integer (nullable = true)
 |-- call_minutes: integer (nullable = true)
 |-- sms_count: integer (nullable = true)

root
 |-- payment_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- bill_month: timestamp (nullable = true)
 |-- amount_paid: integer (nullable = true)
 |-- payment_mode: string (nullable = true)
 |-- payment_status: string (nullable = true)

root
 |-- data_limit_gb: long (nullable = true)
 |-- features: struct (nullable = true)
 |    |-- ott_included: boolean (nullable = true)
 |

In [4]:
!zip -r bronze.zip bronze
files.download('bronze.zip')

  adding: bronze/ (stored 0%)
  adding: bronze/payments/ (stored 0%)
  adding: bronze/payments/part-00000-1b558d0e-61be-490d-965d-ec96020b02ad-c000.snappy.parquet (deflated 48%)
  adding: bronze/payments/_SUCCESS (stored 0%)
  adding: bronze/payments/.part-00000-1b558d0e-61be-490d-965d-ec96020b02ad-c000.snappy.parquet.crc (stored 0%)
  adding: bronze/payments/._SUCCESS.crc (stored 0%)
  adding: bronze/plans/ (stored 0%)
  adding: bronze/plans/_SUCCESS (stored 0%)
  adding: bronze/plans/.part-00000-12c1599c-2dfb-4f53-a9c7-6edc11d1e225-c000.snappy.parquet.crc (stored 0%)
  adding: bronze/plans/part-00000-12c1599c-2dfb-4f53-a9c7-6edc11d1e225-c000.snappy.parquet (deflated 55%)
  adding: bronze/plans/._SUCCESS.crc (stored 0%)
  adding: bronze/usage/ (stored 0%)
  adding: bronze/usage/part-00000-c16db74f-159c-48bd-8b61-96e034c83073-c000.snappy.parquet (deflated 52%)
  adding: bronze/usage/_SUCCESS (stored 0%)
  adding: bronze/usage/.part-00000-c16db74f-159c-48bd-8b61-96e034c83073-c000.snappy

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [8]:
from functools import reduce

customers_df.filter(col('plan_id').isNull()).show()
usage_df.filter(col('data_used_gb').isNull()).show()
payments_df.filter(col('amount_paid').isNull()).show()
payments_df.filter(col('payment_mode').isNull()).show()

usage_df = usage_df.na.fill({
    'data_used_gb': 0
})

payments_df = payments_df.na.fill({
    'amount_paid': 0,
    'payment_mode': 'Not Provided'
})

customers_df = customers_df.na.fill({
    'plan_id': 'UNKNOWN'
})

def data_quality(df):

  condition = reduce(
      lambda a,b : a|b,
      [col(c).isNull() for c in df.columns]
      )

  return df.withColumn(
      'data_quality_status',
      when(condition, 'Incomplete')
      .otherwise('Complete')
  )

data_quality(customers_df).show()
data_quality(usage_df).show()
data_quality(payments_df).show()
data_quality(plans_df).show()


+-----------+-------------+---------+---------+---+------+-------+------+
|customer_id|customer_name|     city|    state|age|gender|plan_id|status|
+-----------+-------------+---------+---------+---+------+-------+------+
|        112|  Ayesha Khan|Hyderabad|Telangana| 28|Female|   NULL|Active|
+-----------+-------------+---------+---------+---+------+-------+------+

+--------+-----------+-------------------+------------+------------+---------+
|usage_id|customer_id|        usage_month|data_used_gb|call_minutes|sms_count|
+--------+-----------+-------------------+------------+------------+---------+
|    1015|        105|2026-02-01 00:00:00|        NULL|        1450|      210|
+--------+-----------+-------------------+------------+------------+---------+

+----------+-----------+-------------------+-----------+------------+--------------+
|payment_id|customer_id|         bill_month|amount_paid|payment_mode|payment_status|
+----------+-----------+-------------------+-----------+-------

In [9]:
!mkdir -p silver

customers_df.write.mode('overwrite').parquet('silver/customers')
usage_df.write.mode('overwrite').parquet('silver/usage')
payments_df.write.mode('overwrite').parquet('silver/payments')
plans_df.write.mode('overwrite').parquet('silver/plans')

!zip -r silver.zip silver
files.download('silver.zip')

  adding: silver/ (stored 0%)
  adding: silver/payments/ (stored 0%)
  adding: silver/payments/_SUCCESS (stored 0%)
  adding: silver/payments/part-00000-757b698c-385d-4493-b8ca-45fe58acfd52-c000.snappy.parquet (deflated 47%)
  adding: silver/payments/.part-00000-757b698c-385d-4493-b8ca-45fe58acfd52-c000.snappy.parquet.crc (stored 0%)
  adding: silver/payments/._SUCCESS.crc (stored 0%)
  adding: silver/plans/ (stored 0%)
  adding: silver/plans/_SUCCESS (stored 0%)
  adding: silver/plans/part-00000-f574dd64-62f2-491f-b29e-3a59e35422ca-c000.snappy.parquet (deflated 55%)
  adding: silver/plans/._SUCCESS.crc (stored 0%)
  adding: silver/plans/.part-00000-f574dd64-62f2-491f-b29e-3a59e35422ca-c000.snappy.parquet.crc (stored 0%)
  adding: silver/usage/ (stored 0%)
  adding: silver/usage/_SUCCESS (stored 0%)
  adding: silver/usage/.part-00000-96c0d6be-8ebe-40f5-bf44-e67bb679eefe-c000.snappy.parquet.crc (stored 0%)
  adding: silver/usage/._SUCCESS.crc (stored 0%)
  adding: silver/usage/part-0000

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [12]:
flat_plans_df = plans_df.select(
    'data_limit_gb',
    'features.ott_included',
    'features.roaming',
    'features.unlimited_calls',
    'monthly_fee',
    'plan_id',
    'plan_name'
)
flat_plans_df.show()

+-------------+------------+-------------+---------------+-----------+-------+------------+
|data_limit_gb|ott_included|      roaming|unlimited_calls|monthly_fee|plan_id|   plan_name|
+-------------+------------+-------------+---------------+-----------+-------+------------+
|           50|       false|     National|           true|        499|   P101| Smart Basic|
|           75|        true|     National|           true|        799|   P102|  Smart Plus|
|           25|       false|         NULL|          false|        299|   P103|Budget Saver|
|          100|        true|International|           true|       1199|   P104| Premium Max|
+-------------+------------+-------------+---------------+-----------+-------+------------+



In [14]:
flat_plans_df.select('plan_id', 'unlimited_calls').show()
flat_plans_df.select('plan_id', 'ott_included').show()
flat_plans_df.select('plan_id', 'roaming').show()

+-------+---------------+
|plan_id|unlimited_calls|
+-------+---------------+
|   P101|           true|
|   P102|           true|
|   P103|          false|
|   P104|           true|
+-------+---------------+

+-------+------------+
|plan_id|ott_included|
+-------+------------+
|   P101|       false|
|   P102|        true|
|   P103|       false|
|   P104|        true|
+-------+------------+

+-------+-------------+
|plan_id|      roaming|
+-------+-------------+
|   P101|     National|
|   P102|     National|
|   P103|         NULL|
|   P104|International|
+-------+-------------+



In [15]:
flat_plans_df = flat_plans_df.na.fill({
    'roaming': 'Not Available'
})
flat_plans_df.write.mode('overwrite').parquet('flat_plans_df')

!zip -r flat_plans_df.zip flat_plans_df

  adding: flat_plans_df/ (stored 0%)
  adding: flat_plans_df/_SUCCESS (stored 0%)
  adding: flat_plans_df/.part-00000-0b0a2d2f-5c72-4aa3-92e3-4f37f17d1ee7-c000.snappy.parquet.crc (stored 0%)
  adding: flat_plans_df/._SUCCESS.crc (stored 0%)
  adding: flat_plans_df/part-00000-0b0a2d2f-5c72-4aa3-92e3-4f37f17d1ee7-c000.snappy.parquet (deflated 53%)


In [16]:
files.download('flat_plans_df.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [19]:
customers_df.join(
    plans_df,
    'plan_id',
    'left'
).show()
customers_df.join(
    usage_df,
    'customer_id',
    'left'
).show()
customers_df.join(
    payments_df,
    'customer_id',
    'left'
).show()

+-------+-----------+-------------+---------+-----------+---+------+--------+-------------+--------------------+-----------+------------+
|plan_id|customer_id|customer_name|     city|      state|age|gender|  status|data_limit_gb|            features|monthly_fee|   plan_name|
+-------+-----------+-------------+---------+-----------+---+------+--------+-------------+--------------------+-----------+------------+
|   P101|        101| Rahul Sharma|Hyderabad|  Telangana| 35|  Male|  Active|           50|{false, National,...|        499| Smart Basic|
|   P102|        102|  Priya Reddy|Bangalore|  Karnataka| 29|Female|  Active|           75|{true, National, ...|        799|  Smart Plus|
|   P103|        103|   Amit Kumar|   Mumbai|Maharashtra| 42|  Male|Inactive|           25|{false, NULL, false}|        299|Budget Saver|
|   P101|        104|  Sneha Patel|  Chennai| Tamil Nadu| 31|Female|  Active|           50|{false, National,...|        499| Smart Basic|
|   P104|        105|   Farhan Ali

In [21]:
customers_usage_billing_df = customers_df.join(
    usage_df,
    'customer_id',
    'left'
).join(
    payments_df,
    'customer_id',
    'left'
).join(
    plans_df,
    'plan_id',
    'left'
)
customers_usage_billing_df.show()

+-------+-----------+-------------+---------+-----------+---+------+--------+--------+-------------------+------------+------------+---------+----------+-------------------+-----------+------------+--------------+-------------+--------------------+-----------+------------+
|plan_id|customer_id|customer_name|     city|      state|age|gender|  status|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|payment_id|         bill_month|amount_paid|payment_mode|payment_status|data_limit_gb|            features|monthly_fee|   plan_name|
+-------+-----------+-------------+---------+-----------+---+------+--------+--------+-------------------+------------+------------+---------+----------+-------------------+-----------+------------+--------------+-------------+--------------------+-----------+------------+
|   P101|        101| Rahul Sharma|Hyderabad|  Telangana| 35|  Male|  Active|    1012|2026-02-01 00:00:00|          50|        1000|      130|      5012|2026-02-01 00:00:00|     

In [22]:
customers_df.join(
    plans_df.select('plan_id').distinct(),
    'plan_id',
    'left_anti'
).show()

usage_df.join(
    customers_df.select('customer_id').distinct(),
    'customer_id',
    'left_anti'
).show()

payments_df.join(
    customers_df.select('customer_id').distinct(),
    'customer_id',
    'left_anti'
).show()

+-------+-----------+-------------+---------+-----------+---+------+------+
|plan_id|customer_id|customer_name|     city|      state|age|gender|status|
+-------+-----------+-------------+---------+-----------+---+------+------+
|   P105|        111|   Ravi Kumar|   Mumbai|Maharashtra| 45|  Male|Active|
|UNKNOWN|        112|  Ayesha Khan|Hyderabad|  Telangana| 28|Female|Active|
+-------+-----------+-------------+---------+-----------+---+------+------+

+-----------+--------+-------------------+------------+------------+---------+
|customer_id|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|
+-----------+--------+-------------------+------------+------------+---------+
|        120|    1011|2026-01-01 00:00:00|          60|        1300|      140|
+-----------+--------+-------------------+------------+------------+---------+

+-----------+----------+----------+-----------+------------+--------------+
|customer_id|payment_id|bill_month|amount_paid|payment_mode|payment_sta

In [30]:
usage_cat_df = usage_df.withColumn(
    'usage_category',
    when(col('data_used_gb') >= 70, 'Heavy User')
    .when(col('data_used_gb') >= 30, 'Medium User')
    .otherwise('Low User')
)
usage_cat_df.show()

payments_df.withColumn(
    'payment_category',
    when(col('amount_paid') >= 1000, 'High Payment')
    .when(col('amount_paid') >= 500, 'Medium Payment')
    .otherwise('Low Payment')
).show()

churn_risk_df = customers_df.join(
    payments_df,
    'customer_id',
    'left'
).join(
    usage_df,
    'customer_id',
    'left'
).withColumn(
    'churn_risk',
    when((col('status')=='Inactive') | (col('payment_status')!='Success'), 'High Risk')
    .when(col('data_used_gb')<15, 'Medium Risk')
    .otherwise('Low Risk')
)
churn_risk_df.show()

over_usage_df = usage_df.join(
    customers_df.select('customer_id', 'plan_id'),
    'customer_id',
    'left'
).join(
    plans_df,
    'plan_id',
    'left'
).withColumn(
    'over_usage_gb',
    when((col('data_used_gb') - col('data_limit_gb'))<=0, 0)
    .otherwise(col('data_used_gb') - col('data_limit_gb'))
).select('customer_id', 'plan_id', 'over_usage_gb')
over_usage_df.show()

over_usage_df.withColumn(
    'over_usage_flag',
    when(col('over_usage_gb')>0, 'Yes')
    .otherwise('No')
).show()


+--------+-----------+-------------------+------------+------------+---------+--------------+
|usage_id|customer_id|        usage_month|data_used_gb|call_minutes|sms_count|usage_category|
+--------+-----------+-------------------+------------+------------+---------+--------------+
|    1001|        101|2026-01-01 00:00:00|          45|         900|      120|   Medium User|
|    1002|        102|2026-01-01 00:00:00|          30|         600|       80|   Medium User|
|    1003|        103|2026-01-01 00:00:00|          12|         250|       40|      Low User|
|    1004|        104|2026-01-01 00:00:00|          55|        1100|      150|   Medium User|
|    1005|        105|2026-01-01 00:00:00|          75|        1500|      200|    Heavy User|
|    1006|        106|2026-01-01 00:00:00|          28|         500|       60|      Low User|
|    1007|        107|2026-01-01 00:00:00|          10|         200|       20|      Low User|
|    1008|        108|2026-01-01 00:00:00|          80|     

In [31]:
customers_df.groupBy('city').agg(count('customer_id').alias('customer count')).show()
customers_df.groupBy('state').agg(count('customer_id').alias('customer count')).show()
customers_df.groupBy('plan_id').agg(count('customer_id').alias('customer count')).show()
usage_cat_df.groupBy('usage_category').agg(count('*').alias('cusotomer count')).show()
churn_risk_df.groupBy('churn_risk').agg(count('*').alias('customer count')).show()

+---------+--------------+
|     city|customer count|
+---------+--------------+
|Bangalore|             2|
|    Kochi|             1|
|  Chennai|             1|
|   Mumbai|             2|
|     Pune|             1|
|    Delhi|             2|
|Hyderabad|             3|
+---------+--------------+

+-----------+--------------+
|      state|customer count|
+-----------+--------------+
|  Karnataka|             2|
|     Kerala|             1|
| Tamil Nadu|             1|
|      Delhi|             2|
|  Telangana|             3|
|Maharashtra|             3|
+-----------+--------------+

+-------+--------------+
|plan_id|customer count|
+-------+--------------+
|   P105|             1|
|   P102|             3|
|UNKNOWN|             1|
|   P103|             2|
|   P104|             2|
|   P101|             3|
+-------+--------------+

+--------------+---------------+
|usage_category|cusotomer count|
+--------------+---------------+
|   Medium User|              9|
|    Heavy User|            

In [34]:
plans_df.join(
    customers_df.select('customer_id', 'plan_id'),
    'plan_id',
    'left'
).join(
    usage_df,
    'customer_id',
    'left'
).groupby('plan_id', 'plan_name').agg(sum('data_used_gb').alias('total data')).show()

+-------+------------+----------+
|plan_id|   plan_name|total data|
+-------+------------+----------+
|   P103|Budget Saver|        22|
|   P101| Smart Basic|       256|
|   P104| Premium Max|       155|
|   P102|  Smart Plus|       124|
+-------+------------+----------+



In [36]:
plans_df.join(
    customers_df.select('customer_id', 'plan_id'),
    'plan_id',
    'left'
).join(
    usage_df,
    'customer_id',
    'left'
).groupby('plan_id', 'plan_name').agg(avg('data_used_gb').alias('avg data')).show()

+-------+------------+------------------+
|plan_id|   plan_name|          avg data|
+-------+------------+------------------+
|   P103|Budget Saver|              11.0|
|   P101| Smart Basic|              51.2|
|   P104| Premium Max|51.666666666666664|
|   P102|  Smart Plus|              31.0|
+-------+------------+------------------+



In [37]:
customers_df.join(
    usage_df,
    'customer_id',
    'left'
).groupBy('city').agg(sum('call_minutes')).show()

+---------+-----------------+
|     city|sum(call_minutes)|
+---------+-----------------+
|Bangalore|             2200|
|    Kochi|             1600|
|  Chennai|             2300|
|   Mumbai|              250|
|     Pune|              500|
|    Delhi|             3650|
|Hyderabad|             2100|
+---------+-----------------+



In [38]:
customers_df.join(
    usage_df,
    'customer_id',
    'left'
).groupBy('state').agg(sum('sms_count')).show()

+-----------+--------------+
|      state|sum(sms_count)|
+-----------+--------------+
|  Karnataka|           265|
|     Kerala|           250|
| Tamil Nadu|           310|
|      Delhi|           500|
|  Telangana|           270|
|Maharashtra|           100|
+-----------+--------------+



In [40]:
payments_df.filter(col('payment_status')=='Success').agg(sum('amount_paid').alias('revenue')).show()

+-------+
|revenue|
+-------+
|   8089|
+-------+



In [41]:
customers_df.join(
    payments_df,
    'customer_id',
    'left'
).groupby('city').agg(sum('amount_paid').alias('revenue')).show()

+---------+-------+
|     city|revenue|
+---------+-------+
|Bangalore|   2097|
|    Kochi|   1199|
|  Chennai|    998|
|   Mumbai|    299|
|     Pune|    799|
|    Delhi|   3197|
|Hyderabad|   1297|
+---------+-------+



In [43]:
plans_df.join(
    customers_df.select('plan_id', 'customer_id'),
    'plan_id',
    'left'
).join(
    payments_df,
    'customer_id',
    'left'
).groupBy('plan_id').agg(sum('amount_paid').alias('revenue')).show()

+-------+-------+
|plan_id|revenue|
+-------+-------+
|   P102|   3196|
|   P103|    598|
|   P104|   3597|
|   P101|   2495|
+-------+-------+



In [44]:
payments_df.groupby('payment_mode').agg(sum('amount_paid')).show()

+------------+----------------+
|payment_mode|sum(amount_paid)|
+------------+----------------+
|        Card|            3696|
|        Cash|             598|
|Not Provided|            1199|
|         UPI|            4393|
+------------+----------------+



In [46]:
plans_df.join(
    customers_df.select('plan_id', 'customer_id'),
    'plan_id',
    'left'
).join(
    payments_df,
    'customer_id',
    'left'
).groupBy('plan_id').agg(sum('amount_paid').alias('revenue')).orderBy(col('revenue').desc()).show(1)

+-------+-------+
|plan_id|revenue|
+-------+-------+
|   P104|   3597|
+-------+-------+
only showing top 1 row


In [48]:
customers_df.join(
    payments_df,
    'customer_id',
    'left'
).groupby('city').agg(sum('amount_paid').alias('revenue')).orderBy(col('revenue').desc()).show(1)

+-----+-------+
| city|revenue|
+-----+-------+
|Delhi|   3197|
+-----+-------+
only showing top 1 row


In [50]:
from pyspark.sql.window import Window as win

win_spec = win.orderBy(col('data_used_gb').desc())
customers_df.join(
    usage_df,
    'customer_id',
    'left'
).withColumn(
    'usage_rank',
    rank().over(win_spec)
).show()

+-----------+-------------+---------+-----------+---+------+-------+--------+--------+-------------------+------------+------------+---------+----------+
|customer_id|customer_name|     city|      state|age|gender|plan_id|  status|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|usage_rank|
+-----------+-------------+---------+-----------+---+------+-------+--------+--------+-------------------+------------+------------+---------+----------+
|        108|   Meera Nair|    Kochi|     Kerala| 48|Female|   P104|  Active|    1008|2026-01-01 00:00:00|          80|        1600|      250|         1|
|        105|   Farhan Ali|    Delhi|      Delhi| 55|  Male|   P104|  Active|    1005|2026-01-01 00:00:00|          75|        1500|      200|         2|
|        104|  Sneha Patel|  Chennai| Tamil Nadu| 31|Female|   P101|  Active|    1014|2026-02-01 00:00:00|          58|        1200|      160|         3|
|        104|  Sneha Patel|  Chennai| Tamil Nadu| 31|Female|   P101|  Active

In [52]:
win_spec = win.orderBy(col('total paid').desc())

customers_df.join(
    payments_df,
    'customer_id',
    'left'
).groupby('customer_id').agg(sum('amount_paid').alias('total paid')).withColumn(
    'rank',
    rank().over(win_spec)
).show()

+-----------+----------+----+
|customer_id|total paid|rank|
+-----------+----------+----+
|        105|      2398|   1|
|        102|      1598|   2|
|        108|      1199|   3|
|        101|       998|   4|
|        104|       998|   4|
|        110|       799|   6|
|        106|       799|   6|
|        109|       499|   8|
|        103|       299|   9|
|        107|       299|   9|
|        112|         0|  11|
|        111|      NULL|  12|
+-----------+----------+----+



In [53]:
win_spec = win.orderBy(col('data_used_gb').desc())
customers_df.join(
    usage_df,
    'customer_id',
    'left'
).withColumn(
    'usage_rank',
    rank().over(win_spec)
).show(3)

+-----------+-------------+-------+----------+---+------+-------+------+--------+-------------------+------------+------------+---------+----------+
|customer_id|customer_name|   city|     state|age|gender|plan_id|status|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|usage_rank|
+-----------+-------------+-------+----------+---+------+-------+------+--------+-------------------+------------+------------+---------+----------+
|        108|   Meera Nair|  Kochi|    Kerala| 48|Female|   P104|Active|    1008|2026-01-01 00:00:00|          80|        1600|      250|         1|
|        105|   Farhan Ali|  Delhi|     Delhi| 55|  Male|   P104|Active|    1005|2026-01-01 00:00:00|          75|        1500|      200|         2|
|        104|  Sneha Patel|Chennai|Tamil Nadu| 31|Female|   P101|Active|    1014|2026-02-01 00:00:00|          58|        1200|      160|         3|
+-----------+-------------+-------+----------+---+------+-------+------+--------+-------------------+-----

In [54]:
win_spec = win.orderBy(col('total paid').desc())

customers_df.join(
    payments_df,
    'customer_id',
    'left'
).groupby('customer_id').agg(sum('amount_paid').alias('total paid')).withColumn(
    'rank',
    rank().over(win_spec)
).show(3)

+-----------+----------+----+
|customer_id|total paid|rank|
+-----------+----------+----+
|        105|      2398|   1|
|        102|      1598|   2|
|        108|      1199|   3|
+-----------+----------+----+
only showing top 3 rows


In [56]:
win_spec = win.partitionBy('city').orderBy(col('total paid').desc())

customers_df.join(
    payments_df,
    'customer_id',
    'left'
).groupby('city').agg(sum('amount_paid').alias('total paid')).withColumn(
    'rank',
    rank().over(win_spec)
).show()

+---------+----------+----+
|     city|total paid|rank|
+---------+----------+----+
|Bangalore|      2097|   1|
|  Chennai|       998|   1|
|    Delhi|      3197|   1|
|Hyderabad|      1297|   1|
|    Kochi|      1199|   1|
|   Mumbai|       299|   1|
|     Pune|       799|   1|
+---------+----------+----+



In [60]:
win_spec = win.partitionBy('plan_name').orderBy(col('revenue').desc())

customers_df.join(
    plans_df,
    'plan_id',
    'left'
).join(
    payments_df,
    'customer_id',
    'left'
).groupBy('customer_id','plan_name').agg(sum('amount_paid').alias('revenue')).withColumn(
    'rank',
    rank().over(win_spec)
).filter(col('rank')==1).show()

+-----------+------------+-------+----+
|customer_id|   plan_name|revenue|rank|
+-----------+------------+-------+----+
|        112|        NULL|      0|   1|
|        107|Budget Saver|    299|   1|
|        103|Budget Saver|    299|   1|
|        105| Premium Max|   2398|   1|
|        104| Smart Basic|    998|   1|
|        101| Smart Basic|    998|   1|
|        102|  Smart Plus|   1598|   1|
+-----------+------------+-------+----+



In [64]:
monthly_revenue_df = payments_df.withColumn(
    'month',
    month('bill_month')
).groupBy('month').agg(sum('amount_paid').alias('monthly_revenue'))

win_spec = win.orderBy('month')

monthly_revenue_df.withColumn(
    'running_total_revenue',
    sum('monthly_revenue').over(win_spec)
).show()

+-----+---------------+---------------------+
|month|monthly_revenue|running_total_revenue|
+-----+---------------+---------------------+
|    1|           6890|                 6890|
|    2|           2996|                 9886|
+-----+---------------+---------------------+



In [70]:
win_spec = win.orderBy('usage_month')
usage_df.withColumns({
    'prev': lag('data_used_gb').over(win_spec),
    'nex' : lead('data_used_gb').over(win_spec)
}).orderBy('usage_month').show()

+--------+-----------+-------------------+------------+------------+---------+----+----+
|usage_id|customer_id|        usage_month|data_used_gb|call_minutes|sms_count|prev| nex|
+--------+-----------+-------------------+------------+------------+---------+----+----+
|    1001|        101|2026-01-01 00:00:00|          45|         900|      120|NULL|  30|
|    1002|        102|2026-01-01 00:00:00|          30|         600|       80|  45|  12|
|    1003|        103|2026-01-01 00:00:00|          12|         250|       40|  30|  55|
|    1004|        104|2026-01-01 00:00:00|          55|        1100|      150|  12|  75|
|    1005|        105|2026-01-01 00:00:00|          75|        1500|      200|  55|  28|
|    1006|        106|2026-01-01 00:00:00|          28|         500|       60|  75|  10|
|    1007|        107|2026-01-01 00:00:00|          10|         200|       20|  28|  80|
|    1008|        108|2026-01-01 00:00:00|          80|        1600|      250|  10|  48|
|    1009|        109

In [ ]:
win_spec = win.partitionBy('customer_id').orderBy('usage_month')

customers_df.join(
    usage_df,
    'customer_id',
    'left'
).withColumn(
    'increased',

)

In [72]:
win_spec = win.partitionBy('customer_id').orderBy('usage_month')

usage_df.withColumn(
    'previous_usage',
    lag('data_used_gb').over(win_spec)
).filter(col('data_used_gb') > col('previous_usage')).show()

+--------+-----------+-------------------+------------+------------+---------+--------------+
|usage_id|customer_id|        usage_month|data_used_gb|call_minutes|sms_count|previous_usage|
+--------+-----------+-------------------+------------+------------+---------+--------------+
|    1012|        101|2026-02-01 00:00:00|          50|        1000|      130|            45|
|    1013|        102|2026-02-01 00:00:00|          34|         650|       85|            30|
|    1014|        104|2026-02-01 00:00:00|          58|        1200|      160|            55|
+--------+-----------+-------------------+------------+------------+---------+--------------+



In [73]:
customers_df.createOrReplaceTempView('customers')
usage_df.createOrReplaceTempView('usage')
payments_df.createOrReplaceTempView('payments')
plans_df.createOrReplaceTempView('plans')

In [82]:
spark.sql("""
  select * from customers where status='Active'
""").show()
spark.sql("""
  select city, count(*) as customer_count from customers
  group by city
""").show()
spark.sql("""
  select p.plan_id, p.plan_name, sum(amount_paid) as revenue from customers c
  join plans p on c.plan_id = p.plan_id
  join payments pa on pa.customer_id = c.customer_id
  group by p.plan_id, p.plan_name
""").show()

+-----------+-------------+---------+-----------+---+------+-------+------+
|customer_id|customer_name|     city|      state|age|gender|plan_id|status|
+-----------+-------------+---------+-----------+---+------+-------+------+
|        101| Rahul Sharma|Hyderabad|  Telangana| 35|  Male|   P101|Active|
|        102|  Priya Reddy|Bangalore|  Karnataka| 29|Female|   P102|Active|
|        104|  Sneha Patel|  Chennai| Tamil Nadu| 31|Female|   P101|Active|
|        105|   Farhan Ali|    Delhi|      Delhi| 55|  Male|   P104|Active|
|        106|   Neha Singh|     Pune|Maharashtra| 38|Female|   P102|Active|
|        108|   Meera Nair|    Kochi|     Kerala| 48|Female|   P104|Active|
|        109|    Kiran Rao|Bangalore|  Karnataka| 33|  Male|   P101|Active|
|        110|  Nisha Reddy|    Delhi|      Delhi| 41|Female|   P102|Active|
|        111|   Ravi Kumar|   Mumbai|Maharashtra| 45|  Male|   P105|Active|
|        112|  Ayesha Khan|Hyderabad|  Telangana| 28|Female|UNKNOWN|Active|
+-----------

In [83]:
spark.sql("""
    select * from customers c
    join usage u on c.customer_id = u.customer_id
    where u.data_used_gb > 70
""").show()


+-----------+-------------+-----+------+---+------+-------+------+--------+-----------+-------------------+------------+------------+---------+
|customer_id|customer_name| city| state|age|gender|plan_id|status|usage_id|customer_id|        usage_month|data_used_gb|call_minutes|sms_count|
+-----------+-------------+-----+------+---+------+-------+------+--------+-----------+-------------------+------------+------------+---------+
|        105|   Farhan Ali|Delhi| Delhi| 55|  Male|   P104|Active|    1005|        105|2026-01-01 00:00:00|          75|        1500|      200|
|        108|   Meera Nair|Kochi|Kerala| 48|Female|   P104|Active|    1008|        108|2026-01-01 00:00:00|          80|        1600|      250|
+-----------+-------------+-----+------+---+------+-------+------+--------+-----------+-------------------+------------+------------+---------+



In [87]:
churn_risk_df.createOrReplaceTempView('churn_risk_table')

spark.sql("""
  select * from customers
  where customer_id in (
    select distinct customer_id from churn_risk_table
    where churn_risk == 'High Risk'
  )
""").show()

+-----------+-------------+---------+-----------+---+------+-------+--------+
|customer_id|customer_name|     city|      state|age|gender|plan_id|  status|
+-----------+-------------+---------+-----------+---+------+-------+--------+
|        103|   Amit Kumar|   Mumbai|Maharashtra| 42|  Male|   P103|Inactive|
|        105|   Farhan Ali|    Delhi|      Delhi| 55|  Male|   P104|  Active|
|        107|  Arjun Verma|Hyderabad|  Telangana| 26|  Male|   P103|Inactive|
+-----------+-------------+---------+-----------+---+------+-------+--------+



In [89]:
spark.sql("""
  select * from customers
  where plan_id not in (
    select distinct plan_id from plans
  )
""").show()

+-----------+-------------+---------+-----------+---+------+-------+------+
|customer_id|customer_name|     city|      state|age|gender|plan_id|status|
+-----------+-------------+---------+-----------+---+------+-------+------+
|        111|   Ravi Kumar|   Mumbai|Maharashtra| 45|  Male|   P105|Active|
|        112|  Ayesha Khan|Hyderabad|  Telangana| 28|Female|UNKNOWN|Active|
+-----------+-------------+---------+-----------+---+------+-------+------+



In [92]:
spark.sql("""
  select * from payments
  where payment_status in ('Pending', 'Failed')
""").show()

+----------+-----------+-------------------+-----------+------------+--------------+
|payment_id|customer_id|         bill_month|amount_paid|payment_mode|payment_status|
+----------+-----------+-------------------+-----------+------------+--------------+
|      5003|        103|2026-01-01 00:00:00|        299|        Cash|        Failed|
|      5007|        107|2026-01-01 00:00:00|        299|        Cash|       Pending|
|      5015|        105|2026-02-01 00:00:00|       1199|Not Provided|       Pending|
+----------+-----------+-------------------+-----------+------------+--------------+



In [95]:
spark.sql("""
  select c.* from customers c
  join(
    select customer_id from usage
    group by customer_id
    order by sum(data_used_gb) desc
    limit 5
  ) u
  on c.customer_id = u.customer_id
""").show()

+-----------+-------------+---------+----------+---+------+-------+------+
|customer_id|customer_name|     city|     state|age|gender|plan_id|status|
+-----------+-------------+---------+----------+---+------+-------+------+
|        101| Rahul Sharma|Hyderabad| Telangana| 35|  Male|   P101|Active|
|        102|  Priya Reddy|Bangalore| Karnataka| 29|Female|   P102|Active|
|        104|  Sneha Patel|  Chennai|Tamil Nadu| 31|Female|   P101|Active|
|        105|   Farhan Ali|    Delhi|     Delhi| 55|  Male|   P104|Active|
|        108|   Meera Nair|    Kochi|    Kerala| 48|Female|   P104|Active|
+-----------+-------------+---------+----------+---+------+-------+------+



In [97]:
spark.sql("""
  select payment_mode, sum(amount_paid) as revenue from payments
  group by payment_mode
""").show()

+------------+-------+
|payment_mode|revenue|
+------------+-------+
|        Card|   3696|
|        Cash|    598|
|Not Provided|   1199|
|         UPI|   4393|
+------------+-------+



In [98]:
!mkdir -p gold

customers_usage_billing_df.write.mode('overwrite').parquet('gold/customers_usage_billing_df')

usage_df.write.mode('overwrite').partitionBy('usage_month').parquet('gold/usageByMonth')


In [99]:
incremental_usage = spark.createDataFrame([
    ('C001', '2026-03', 45),
    ('C002', '2026-03', 60),
    ('C003', '2026-03', 30)
], ['customer_id', 'usage_month', 'data_used_gb'])

incremental_usage.write.mode('overwrite').json('incremental_usage_2026_03')

In [100]:
incremental_usage_df = spark.read.json('incremental_usage_2026_03')
incremental_usage_df.show()

+-----------+------------+-----------+
|customer_id|data_used_gb|usage_month|
+-----------+------------+-----------+
|       C002|          60|    2026-03|
|       C003|          30|    2026-03|
|       C001|          45|    2026-03|
+-----------+------------+-----------+



In [101]:
before_count = spark.read.parquet('silver/usage').count()

In [102]:
incremental_usage_df.write.mode('append').parquet('silver/usage')

In [103]:
customer_usage_summary = usage_df.groupBy('customer_id').agg(
    sum('data_used_gb').alias('total_usage_gb')
)

customer_usage_summary.show()

+-----------+--------------+
|customer_id|total_usage_gb|
+-----------+--------------+
|        108|            80|
|        101|            95|
|        103|            12|
|        120|            60|
|        107|            10|
|        102|            64|
|        109|            48|
|        105|            75|
|        110|            32|
|        106|            28|
|        104|           113|
+-----------+--------------+



In [104]:
gold_df = customers_df.join(
    usage_df,
    'customer_id',
    'left'
).join(
    payments_df,
    'customer_id',
    'left'
).join(
    flat_plans_df,
    'plan_id',
    'left'
)

gold_df.write.mode('overwrite').partitionBy('usage_month').parquet('gold')

In [106]:
after_count = spark.read.parquet('silver/usage').count()

print("Before Incremental Load :", before_count)
print("After Incremental Load  :", after_count)
print("New Records Added       :", after_count - before_count)

Before Incremental Load : 15
After Incremental Load  : 18
New Records Added       : 3


In [107]:
!zip -r gold.zip gold

from google.colab import files
files.download('gold.zip')

  adding: gold/ (stored 0%)
  adding: gold/_SUCCESS (stored 0%)
  adding: gold/usage_month=2026-02-01 00%3A00%3A00/ (stored 0%)
  adding: gold/usage_month=2026-02-01 00%3A00%3A00/.part-00000-3d603634-2574-42ab-8d21-749c0f1810e1.c000.snappy.parquet.crc (stored 0%)
  adding: gold/usage_month=2026-02-01 00%3A00%3A00/part-00000-3d603634-2574-42ab-8d21-749c0f1810e1.c000.snappy.parquet (deflated 64%)
  adding: gold/usage_month=2026-01-01 00%3A00%3A00/ (stored 0%)
  adding: gold/usage_month=2026-01-01 00%3A00%3A00/.part-00000-3d603634-2574-42ab-8d21-749c0f1810e1.c000.snappy.parquet.crc (stored 0%)
  adding: gold/usage_month=2026-01-01 00%3A00%3A00/part-00000-3d603634-2574-42ab-8d21-749c0f1810e1.c000.snappy.parquet (deflated 62%)
  adding: gold/usage_month=__HIVE_DEFAULT_PARTITION__/ (stored 0%)
  adding: gold/usage_month=__HIVE_DEFAULT_PARTITION__/.part-00000-3d603634-2574-42ab-8d21-749c0f1810e1.c000.snappy.parquet.crc (stored 0%)
  adding: gold/usage_month=__HIVE_DEFAULT_PARTITION__/part-000

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [109]:
customer_usage_summary_df = customers_df.join(
    plans_df,
    'plan_id',
    'left'
).join(
    usage_df,
    'customer_id',
    'left'
).join(
    payments_df,
    'customer_id',
    'left'
).withColumn(
    'over_usage_flag',
    when(col('data_used_gb') > col('data_limit_gb'), 'Yes')
    .otherwise('No')
).withColumn(
    'churn_risk',
    when(
        (col('status') == 'Inactive') |
        (col('payment_status') != 'Success'),
        'High Risk'
    )
    .when(col('data_used_gb') < 15, 'Medium Risk')
    .otherwise('Low Risk')
).select(
    'customer_id',
    'customer_name',
    'city',
    'plan_name',
    'usage_month',
    'data_used_gb',
    'data_limit_gb',
    'over_usage_flag',
    'amount_paid',
    'payment_status',
    'churn_risk'
)

customer_usage_summary_df.show(truncate=False)

+-----------+-------------+---------+------------+-------------------+------------+-------------+---------------+-----------+--------------+-----------+
|customer_id|customer_name|city     |plan_name   |usage_month        |data_used_gb|data_limit_gb|over_usage_flag|amount_paid|payment_status|churn_risk |
+-----------+-------------+---------+------------+-------------------+------------+-------------+---------------+-----------+--------------+-----------+
|101        |Rahul Sharma |Hyderabad|Smart Basic |2026-02-01 00:00:00|50          |50           |No             |499        |Success       |Low Risk   |
|101        |Rahul Sharma |Hyderabad|Smart Basic |2026-02-01 00:00:00|50          |50           |No             |499        |Success       |Low Risk   |
|101        |Rahul Sharma |Hyderabad|Smart Basic |2026-01-01 00:00:00|45          |50           |No             |499        |Success       |Low Risk   |
|101        |Rahul Sharma |Hyderabad|Smart Basic |2026-01-01 00:00:00|45          

In [111]:
from pyspark.sql.functions import countDistinct, sum, avg

plan_performance_df = customers_df.join(
    plans_df,
    'plan_id',
    'left'
).join(
    usage_df,
    'customer_id',
    'left'
).join(
    payments_df,
    'customer_id',
    'left'
).groupBy(
    'plan_name'
).agg(
    countDistinct('customer_id').alias('total_customers'),
    sum('data_used_gb').alias('total_data_usage'),
    avg('data_used_gb').alias('average_data_usage'),
    sum('amount_paid').alias('total_revenue')
)

plan_performance_df.show(truncate=False)

+------------+---------------+----------------+------------------+-------------+
|plan_name   |total_customers|total_data_usage|average_data_usage|total_revenue|
+------------+---------------+----------------+------------------+-------------+
|NULL        |2              |NULL            |NULL              |0            |
|Smart Basic |3              |464             |51.55555555555556 |4491         |
|Budget Saver|2              |22              |11.0              |598          |
|Premium Max |2              |230             |46.0              |5995         |
|Smart Plus  |3              |188             |31.333333333333332|4794         |
+------------+---------------+----------------+------------------+-------------+



In [112]:
city_revenue_df = customers_df.join(
    payments_df,
    'customer_id',
    'left'
).groupBy(
    'city'
).agg(
    countDistinct('customer_id').alias('total_customers'),
    sum('amount_paid').alias('total_revenue'),
    avg('amount_paid').alias('average_payment')
)

city_revenue_df.show(truncate=False)

churn_risk_report_df = customers_df.join(
    plans_df,
    'plan_id',
    'left'
).join(
    payments_df,
    'customer_id',
    'left'
).join(
    usage_df,
    'customer_id',
    'left'
).withColumn(
    'churn_risk',
    when(
        (col('status') == 'Inactive') |
        (col('payment_status') != 'Success'),
        'High Risk'
    )
    .when(
        col('data_used_gb') < 15,
        'Medium Risk'
    )
    .otherwise(
        'Low Risk'
    )
).select(
    'customer_id',
    'customer_name',
    'city',
    'plan_name',
    'payment_status',
    'status',
    'churn_risk'
)

churn_risk_report_df.show(truncate=False)

+---------+---------------+-------------+------------------+
|city     |total_customers|total_revenue|average_payment   |
+---------+---------------+-------------+------------------+
|Bangalore|2              |2097         |699.0             |
|Kochi    |1              |1199         |1199.0            |
|Chennai  |1              |998          |499.0             |
|Mumbai   |2              |299          |299.0             |
|Pune     |1              |799          |799.0             |
|Delhi    |2              |3197         |1065.6666666666667|
|Hyderabad|3              |1297         |324.25            |
+---------+---------------+-------------+------------------+

+-----------+-------------+---------+------------+--------------+--------+-----------+
|customer_id|customer_name|city     |plan_name   |payment_status|status  |churn_risk |
+-----------+-------------+---------+------------+--------------+--------+-----------+
|101        |Rahul Sharma |Hyderabad|Smart Basic |Success       |Ac

In [114]:
over_usage_report_df = customers_df.join(
    plans_df,
    'plan_id',
    'left'
).join(
    usage_df,
    'customer_id',
    'left'
).withColumn(
    'over_usage_gb',
    when(
        (col('data_used_gb') - col('data_limit_gb')) <= 0,
        0
    ).otherwise(
        col('data_used_gb') - col('data_limit_gb')
    )
).select(
    'customer_id',
    'customer_name',
    'plan_name',
    'data_used_gb',
    'data_limit_gb',
    'over_usage_gb'
)

over_usage_report_df.show(truncate=False)

+-----------+-------------+------------+------------+-------------+-------------+
|customer_id|customer_name|plan_name   |data_used_gb|data_limit_gb|over_usage_gb|
+-----------+-------------+------------+------------+-------------+-------------+
|101        |Rahul Sharma |Smart Basic |50          |50           |0            |
|101        |Rahul Sharma |Smart Basic |45          |50           |0            |
|102        |Priya Reddy  |Smart Plus  |34          |75           |0            |
|102        |Priya Reddy  |Smart Plus  |30          |75           |0            |
|103        |Amit Kumar   |Budget Saver|12          |25           |0            |
|104        |Sneha Patel  |Smart Basic |58          |50           |8            |
|104        |Sneha Patel  |Smart Basic |55          |50           |5            |
|105        |Farhan Ali   |Premium Max |0           |100          |0            |
|105        |Farhan Ali   |Premium Max |75          |100          |0            |
|106        |Neh

In [113]:
print('customers_df')
customers_df.printSchema()
print('plans_df')
plans_df.printSchema()
print('payments_df')
payments_df.printSchema()
print('usage_df')
usage_df.printSchema()

customers_df
root
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- plan_id: string (nullable = false)
 |-- status: string (nullable = true)

plans_df
root
 |-- data_limit_gb: long (nullable = true)
 |-- features: struct (nullable = true)
 |    |-- ott_included: boolean (nullable = true)
 |    |-- roaming: string (nullable = true)
 |    |-- unlimited_calls: boolean (nullable = true)
 |-- monthly_fee: long (nullable = true)
 |-- plan_id: string (nullable = true)
 |-- plan_name: string (nullable = true)

payments_df
root
 |-- payment_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- bill_month: timestamp (nullable = true)
 |-- amount_paid: integer (nullable = false)
 |-- payment_mode: string (nullable = false)
 |-- payment_status: string (nullable = true)

usage_df
root
 

In [84]:
churn_risk_df.show()

+-----------+-------------+---------+-----------+---+------+-------+--------+----------+-------------------+-----------+------------+--------------+--------+-------------------+------------+------------+---------+-----------+
|customer_id|customer_name|     city|      state|age|gender|plan_id|  status|payment_id|         bill_month|amount_paid|payment_mode|payment_status|usage_id|        usage_month|data_used_gb|call_minutes|sms_count| churn_risk|
+-----------+-------------+---------+-----------+---+------+-------+--------+----------+-------------------+-----------+------------+--------------+--------+-------------------+------------+------------+---------+-----------+
|        101| Rahul Sharma|Hyderabad|  Telangana| 35|  Male|   P101|  Active|      5012|2026-02-01 00:00:00|        499|        Card|       Success|    1012|2026-02-01 00:00:00|          50|        1000|      130|   Low Risk|
|        101| Rahul Sharma|Hyderabad|  Telangana| 35|  Male|   P101|  Active|      5012|2026-02-

In [61]:
payments_df.show()

+----------+-----------+-------------------+-----------+------------+--------------+
|payment_id|customer_id|         bill_month|amount_paid|payment_mode|payment_status|
+----------+-----------+-------------------+-----------+------------+--------------+
|      5001|        101|2026-01-01 00:00:00|        499|         UPI|       Success|
|      5002|        102|2026-01-01 00:00:00|        799|        Card|       Success|
|      5003|        103|2026-01-01 00:00:00|        299|        Cash|        Failed|
|      5004|        104|2026-01-01 00:00:00|        499|         UPI|       Success|
|      5005|        105|2026-01-01 00:00:00|       1199|        Card|       Success|
|      5006|        106|2026-01-01 00:00:00|        799|         UPI|       Success|
|      5007|        107|2026-01-01 00:00:00|        299|        Cash|       Pending|
|      5008|        108|2026-01-01 00:00:00|       1199|        Card|       Success|
|      5009|        109|2026-01-01 00:00:00|        499|         

In [76]:
customers_df.show()

+-----------+-------------+---------+-----------+---+------+-------+--------+
|customer_id|customer_name|     city|      state|age|gender|plan_id|  status|
+-----------+-------------+---------+-----------+---+------+-------+--------+
|        101| Rahul Sharma|Hyderabad|  Telangana| 35|  Male|   P101|  Active|
|        102|  Priya Reddy|Bangalore|  Karnataka| 29|Female|   P102|  Active|
|        103|   Amit Kumar|   Mumbai|Maharashtra| 42|  Male|   P103|Inactive|
|        104|  Sneha Patel|  Chennai| Tamil Nadu| 31|Female|   P101|  Active|
|        105|   Farhan Ali|    Delhi|      Delhi| 55|  Male|   P104|  Active|
|        106|   Neha Singh|     Pune|Maharashtra| 38|Female|   P102|  Active|
|        107|  Arjun Verma|Hyderabad|  Telangana| 26|  Male|   P103|Inactive|
|        108|   Meera Nair|    Kochi|     Kerala| 48|Female|   P104|  Active|
|        109|    Kiran Rao|Bangalore|  Karnataka| 33|  Male|   P101|  Active|
|        110|  Nisha Reddy|    Delhi|      Delhi| 41|Female|   P

In [115]:
!zip -r gold.zip gold

updating: gold/ (stored 0%)
updating: gold/_SUCCESS (stored 0%)
updating: gold/usage_month=2026-02-01 00%3A00%3A00/ (stored 0%)
updating: gold/usage_month=2026-02-01 00%3A00%3A00/.part-00000-3d603634-2574-42ab-8d21-749c0f1810e1.c000.snappy.parquet.crc (stored 0%)
updating: gold/usage_month=2026-02-01 00%3A00%3A00/part-00000-3d603634-2574-42ab-8d21-749c0f1810e1.c000.snappy.parquet (deflated 64%)
updating: gold/usage_month=2026-01-01 00%3A00%3A00/ (stored 0%)
updating: gold/usage_month=2026-01-01 00%3A00%3A00/.part-00000-3d603634-2574-42ab-8d21-749c0f1810e1.c000.snappy.parquet.crc (stored 0%)
updating: gold/usage_month=2026-01-01 00%3A00%3A00/part-00000-3d603634-2574-42ab-8d21-749c0f1810e1.c000.snappy.parquet (deflated 62%)
updating: gold/usage_month=__HIVE_DEFAULT_PARTITION__/ (stored 0%)
updating: gold/usage_month=__HIVE_DEFAULT_PARTITION__/.part-00000-3d603634-2574-42ab-8d21-749c0f1810e1.c000.snappy.parquet.crc (stored 0%)
updating: gold/usage_month=__HIVE_DEFAULT_PARTITION__/part-000

In [116]:
files.download('gold.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [117]:
!zip -r incremental_usage_2026_03.zip incremental_usage_2026_03
files.download('incremental_usage_2026_03.zip')

  adding: incremental_usage_2026_03/ (stored 0%)
  adding: incremental_usage_2026_03/part-00000-964a43b2-d8fa-40f9-a1ad-9a37bb8b0887-c000.json (deflated 2%)
  adding: incremental_usage_2026_03/_SUCCESS (stored 0%)
  adding: incremental_usage_2026_03/.part-00001-964a43b2-d8fa-40f9-a1ad-9a37bb8b0887-c000.json.crc (stored 0%)
  adding: incremental_usage_2026_03/.part-00000-964a43b2-d8fa-40f9-a1ad-9a37bb8b0887-c000.json.crc (stored 0%)
  adding: incremental_usage_2026_03/._SUCCESS.crc (stored 0%)
  adding: incremental_usage_2026_03/part-00001-964a43b2-d8fa-40f9-a1ad-9a37bb8b0887-c000.json (deflated 43%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>